In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sqlite3

# 0
df = sns.load_dataset('penguins').dropna(subset=['bill_length_mm'])
conn = sqlite3.connect(':memory:')
df.to_sql('penguins', conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

islands = pd.DataFrame({
    'island': ['Biscoe', 'Dream', 'Somewhere'],   # Torgersen навмисно відсутній
    'region': ['South', 'Central', 'Nowhere']
})
islands.to_sql('islands', conn, index=False)

# 1
# 1.1 Всього 342 особини представлені на визначених островах.
q("SELECT COUNT(*) AS N FROM penguins")
# 1.2 Всього всі пінгвіни належать до 3 видів: Adelie, Chinstrap та Gentoo
q("SELECT DISTINCT species FROM penguins")
# 1.3 Всі зазначені пінгвіни проживають на 3 островах: Torgersen, Biscoe, Dream
q("SELECT DISTINCT island FROM penguins")

,island
0,Torgersen
1,Biscoe
2,Dream


In [ ]:
# 2 Отже, середня вага й середня довжина плавця в Adelie становить ~3700,6г та ~190мм відповідно,
# Chinstrap - 3733,1г та 195,8мм, Gentoo - 5076г та 217,2мм.
# Отже Gentoo порівняно більші за інші види у масі та довжині плавця.
# Обмовка*: голий species в SELECT спеціально вставлений для розуміння, які види мають, які габарити.
q("SELECT species, AVG(body_mass_g) AS mid_mass, AVG(flipper_length_mm) AS mid_flip_length FROM penguins GROUP BY species")

,species,mid_mass,mid_flip_length
0,Adelie,3700.662252,189.953642
1,Chinstrap,3733.088235,195.823529
2,Gentoo,5076.016260,217.186992


In [ ]:
# 3 Отже найбільшу середню довжину плавця мають пінгвіни з Південного регіону. Найменше пінгвіни з Targersen без прив'язки до конкретного регіону 
q("SELECT i.region, AVG(p.flipper_length_mm) AS mid_flip_length FROM penguins p LEFT JOIN islands i ON p.island = i.island GROUP BY i.region")

,region,mid_flip_length
0,NaN,191.196078
1,Central,193.072581
2,South,209.706587


In [ ]:
# 4 Види, в яких середня довжина крила менша за загальну середню довжину крила усіх пінгвінів
q("SELECT species, AVG(flipper_length_mm) AS mid_flip_length FROM penguins GROUP BY species HAVING AVG(flipper_length_mm) < (SELECT AVG(flipper_length_mm) FROM penguins)")

,species,mid_flip_length
0,Adelie,189.953642
1,Chinstrap,195.823529


In [67]:
# 5 Середня маса пінгвінів за островами, яка менша за загальну середню масу усіх пінгвінів, живуть вони на островах Dream та Torgersen
q("""
    WITH spec_islands AS (SELECT island, AVG(body_mass_g) as mid_mass FROM penguins GROUP BY island)
    SELECT island, mid_mass FROM spec_islands WHERE mid_mass < (SELECT AVG(body_mass_g) FROM penguins)
""")

,island,mid_mass
0,Dream,3712.903226
1,Torgersen,3706.372549


6. Фінальні висновки:
1) Всього в цій популяції налічується 344 (2 з них не враховані в силу майже повної відсутності даних по ним) особини, які проживають на 3 островах: Dream, Torgersen, Biscoe та поділяються на 3 види: Adelie, Chinstrap та Gentoo.
2) Gentoo помітно більші за середньою масою та довжиною плавця на 1343-1376г та 21-28мм відповідно.
3) В південному регіоні живуть наймасивніші пінгвіни.
4) На островах Dream та Torgersen живуть пінгвіни середня маса яких менша за загальну середню масу усіх пінгвінів.
 